2 Exercise 1 - Implementation of Naive Bayes Algorithm
https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [3]:
#load the dataset
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/Concept and Technology of AI/IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# 1(b) Convert all text to lowercase and remove punctuation
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

ps = PorterStemmer()
stop_words = stopwords.words('english')

clean_reviews = []

for i in range(len(data)):
    review = data['review'][i].lower() # lowercase
    review = re.sub('[^a-zA-Z]', ' ', review) # remove punctuation
    clean_reviews.append(review)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
# 1(c) Tokenize and remove stopwords
processed_reviews = []

for review in clean_reviews:
    words = review.split() # tokenization
    words = [word for word in words if word not in stop_words]
    processed_reviews.append(words)

In [7]:
# 1(d) Apply stemming
final_reviews = []

for words in processed_reviews:
    stemmed_words = [ps.stem(word) for word in words]
    final_reviews.append(" ".join(stemmed_words))

data['clean_review'] = final_reviews

In [8]:
# 2. Split the dataset (80% training, 20% testing)
from sklearn.model_selection import train_test_split

data['sentiment'] = data['sentiment'].map({'positive':1, 'negative':0})

X = data['clean_review']
y = data['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
# 3(a) Bag-of-Words using CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000)
X_train_vec = cv.fit_transform(X_train)
X_test_vec = cv.transform(X_test)

In [10]:
# 3(b) Train Naive Bayes Classifier
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

### Part 2

In [11]:
# 1(a) Accuracy
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.84


In [12]:
# 1(b) Precision, Recall, F1-score
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.85      0.84      4999
           1       0.85      0.83      0.84      5001

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



In [13]:
# 1(c) Confusion Matrix
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))


[[4243  756]
 [ 844 4157]]


In [14]:
# 1(d) ROC-AUC Score
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test_vec)[:,1]
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

ROC-AUC Score: 0.9083423563336943


In [ ]:
# Exercise 3 – Feature Selection using Wrapper Methods
https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

In [16]:
# Part 1: Data Loading and Preprocessing
# 1. Load the Breast Cancer Dataset
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/Concept and Technology of AI/data.csv")
data.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [17]:
# 2. Perform basic exploratory data analysis (EDA)

# 2(a) Dataset information
data.info()

# 2(b) Summary statistics
data.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,0.0
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,NaN
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,NaN
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,NaN
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,NaN
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,NaN
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,NaN
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,NaN


In [18]:
# 3. Check for missing values
data.isnull().sum()

,0
id,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0


In [19]:
# 4. Convert target variable (Diagnosis: M = 1, B = 0)

data['diagnosis'] = data['diagnosis'].map({'M':1, 'B':0})

# Separate features and target
X = data.drop(['id', 'diagnosis', 'Unnamed: 32'], axis=1)
y = data['diagnosis']


In [20]:
# 5. Split the dataset into training (80%) and testing (20%)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [21]:
# Part 2: Apply a Wrapper Method (RFE)
# 1. Use Recursive Feature Elimination (RFE) with Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

model = LogisticRegression(max_iter=5000)

# Select top 5 features
rfe = RFE(model, n_features_to_select=5)
rfe = rfe.fit(X_train, y_train)


In [22]:
# 1(a) Top 5 selected features
selected_features = X_train.columns[rfe.support_]
print("Top 5 Selected Features:")
print(selected_features)


Top 5 Selected Features:
Index(['radius_mean', 'concavity_mean', 'radius_worst', 'compactness_worst',
       'concavity_worst'],
      dtype='object')


In [23]:
# 1(b) Ranking of all features
import pandas as pd

ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Ranking": rfe.ranking_
}).sort_values("Ranking")

ranking


,Feature,Ranking
0,radius_mean,1
6,concavity_mean,1
25,compactness_worst,1
20,radius_worst,1
26,concavity_worst,1
27,concave points_worst,2
28,symmetry_worst,3
5,compactness_mean,4
7,concave points_mean,5
24,smoothness_worst,6


In [24]:
#2: Train Logistic Regression using selected features

# Transform dataset to selected features only
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)

# Train model
model.fit(X_train_rfe, y_train)

# Predict
y_pred_rfe = model.predict(X_test_rfe)
y_prob_rfe = model.predict_proba(X_test_rfe)[:,1]

# Part 3: Model Evaluation (Selected Features)

In [25]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred_rfe))

# Precision, Recall, F1-score
print(classification_report(y_test, y_pred_rfe))

# Confusion Matrix
print(confusion_matrix(y_test, y_pred_rfe))

# ROC-AUC Score
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rfe))

Accuracy: 0.9385964912280702
              precision    recall  f1-score   support

           0       0.92      0.99      0.95        71
           1       0.97      0.86      0.91        43

    accuracy                           0.94       114
   macro avg       0.95      0.92      0.93       114
weighted avg       0.94      0.94      0.94       114

[[70  1]
 [ 6 37]]
ROC-AUC: 0.9885358663609565


In [26]:
# Train using all features
model.fit(X_train, y_train)
y_pred_all = model.predict(X_test)
y_prob_all = model.predict_proba(X_test)[:,1]

print("Accuracy (All Features):", accuracy_score(y_test, y_pred_all))
print("ROC-AUC (All Features):", roc_auc_score(y_test, y_prob_all))

print("Accuracy (Selected Features):", accuracy_score(y_test, y_pred_rfe))
print("ROC-AUC (Selected Features):", roc_auc_score(y_test, y_prob_rfe))

Accuracy (All Features): 0.9649122807017544
ROC-AUC (All Features): 0.9980347199475926
Accuracy (Selected Features): 0.9385964912280702
ROC-AUC (Selected Features): 0.9885358663609565


# Part 4: Experiment with Different Numbers of Features

In [27]:
# Top 3 Features
rfe3 = RFE(model, n_features_to_select=3)
rfe3.fit(X_train, y_train)
print("Top 3 Features:", X_train.columns[rfe3.support_])

# Top 7 Features
rfe7 = RFE(model, n_features_to_select=7)
rfe7.fit(X_train, y_train)
print("Top 7 Features:", X_train.columns[rfe7.support_])

Top 3 Features: Index(['radius_worst', 'compactness_worst', 'concavity_worst'], dtype='object')
Top 7 Features: Index(['radius_mean', 'concavity_mean', 'radius_worst', 'compactness_worst',
       'concavity_worst', 'concave points_worst', 'symmetry_worst'],
      dtype='object')
